# Statistics for Data Science — Teaching Notebook

This notebook is designed as a hands-on teaching resource for the statistics most commonly used in data science. It contains explanations, runnable examples, visualizations, and exercises with suggested answers. 🎯📊

**How to use:** Run the cells in Jupyter Notebook or JupyterLab. Code cells produce live results and visualizations.

---

## Objectives

Learners will:

- Understand and compute descriptive statistics (mean, median, variance, standard deviation, IQR).
- Visualize data using histograms, boxplots, and scatter plots.
- Know common probability distributions (normal, binomial, Poisson) and how to sample from them.
- Perform basic inferential statistics: confidence intervals, t-tests, chi-square test, and simple linear regression.
- Interpret results and assumptions for common tests.

---

## 1 — Setup: imports and sample data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt

# Try to import scipy; if unavailable, we'll note it later
try:
    import scipy.stats as stats
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('scipy available:', SCIPY_AVAILABLE)

## 2 — Descriptive statistics

We'll create a sample dataset (simulated) and compute summary statistics: mean, median, variance, standard deviation, skewness, kurtosis, percentiles.

In [ ]:
# Simulate data: exam scores for 200 students
np.random.seed(42)
scores = np.clip(np.random.normal(loc=70, scale=12, size=200), 0, 100)
df = pd.DataFrame({'score': scores})

df.describe().round(2)

In [ ]:
# Compute additional statistics
from scipy import stats as _sps if False else None

mean = df['score'].mean()
median = df['score'].median()
var = df['score'].var()
std = df['score'].std()
q1 = df['score'].quantile(0.25)
q3 = df['score'].quantile(0.75)
iqr = q3 - q1

# robust measures
cv = std / mean

print(f"mean = {mean:.2f}")
print(f"median = {median:.2f}")
print(f"std = {std:.2f}")
print(f"variance = {var:.2f}")
print(f"IQR = {iqr:.2f}")
print(f"coefficient of variation = {cv:.3f}")

# skewness/kurtosis if scipy available
try:
    import scipy.stats as sps
    print('skewness =', round(sps.skew(df['score']),3))
    print('kurtosis =', round(sps.kurtosis(df['score']),3))
except Exception:
    print('scipy.stats not available; skipping skewness/kurtosis')

### Visualization: histogram + boxplot

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.hist(df['score'], bins=15)
plt.title('Histogram of scores')
plt.xlabel('Score')
plt.ylabel('Frequency')

plt.subplot(1,2,2)
plt.boxplot(df['score'], vert=False)
plt.title('Boxplot of scores')
plt.xlabel('Score')
plt.tight_layout()
plt.show()

## 3 — Probability distributions (samples & PMF/PDF)

Examples: Normal, Binomial, Poisson. We'll sample and visualize.

In [ ]:
# Normal distribution sample
np.random.seed(0)
normal_sample = np.random.normal(loc=0, scale=1, size=1000)

plt.figure()
plt.hist(normal_sample, bins=30)
plt.title('Normal(0,1) sample')
plt.show()

# Binomial distribution (n=10, p=0.3)
binom_sample = np.random.binomial(n=10, p=0.3, size=1000)

plt.figure()
plt.hist(binom_sample, bins=range(0,12))
plt.title('Binomial(n=10, p=0.3) sample')
plt.show()

# Poisson distribution (lambda=3)
pois_sample = np.random.poisson(lam=3, size=1000)
plt.figure()
plt.hist(pois_sample, bins=range(0,12))
plt.title('Poisson(lambda=3) sample')
plt.show()

## 4 — Sampling & Central Limit Theorem (CLT)

Demonstrate sample means approaching normality as sample size increases.

In [ ]:
def sample_means(dist_fn, size, n_samples=1000):
    return [np.mean(dist_fn(size)) for _ in range(n_samples)]

# Distribution: exponential (skewed)
np.random.seed(1)

sizes = [1, 5, 30, 100]
plt.figure(figsize=(10,8))
for i, s in enumerate(sizes, 1):
    means = sample_means(lambda n: np.random.exponential(scale=1.0, size=n), s, n_samples=2000)
    plt.subplot(2,2,i)
    plt.hist(means, bins=30)
    plt.title(f'Sample size = {s}')
plt.tight_layout()
plt.show()

## 5 — Confidence intervals (mean)

Compute a 95% CI for the mean (using t-distribution).

In [ ]:
import math

x = df['score'].values
n = len(x)
mean = x.mean()
se = x.std(ddof=1) / math.sqrt(n)

if SCIPY_AVAILABLE:
    ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=se)
    print(f"95% CI (t) = ({ci[0]:.2f}, {ci[1]:.2f})")
else:
    # approximate using z critical value 1.96
    approx = (mean - 1.96*se, mean + 1.96*se)
    print('scipy not available — using z-approximation')
    print(f"approx 95% CI = ({approx[0]:.2f}, {approx[1]:.2f})")

## 6 — Hypothesis testing: one-sample t-test

Test if the mean score differs from 75 (H0: mu = 75).

In [ ]:
mu0 = 75

if SCIPY_AVAILABLE:
    tstat, pval = stats.ttest_1samp(x, popmean=mu0)
    print('t-statistic =', round(tstat,3))
    print('p-value =', round(pval,4))
else:
    # manual t-test
    mean = x.mean()
    s = x.std(ddof=1)
    tstat = (mean - mu0) / (s/math.sqrt(n))
    # two-sided p approx using normal
    pval = 2*(1 - 0.5*(1 + math.erf(abs(tstat)/math.sqrt(2))))
    print('scipy not available; t-statistic (approx) =', round(tstat,3))
    print('p-value (approx) =', round(pval,4))

alpha = 0.05
if pval < alpha:
    print('Reject H0 at alpha=0.05')
else:
    print('Fail to reject H0 at alpha=0.05')

## 7 — Two-sample t-test (independent)

Simulate two groups (control vs treatment) and test difference in means.

In [ ]:
# Simulate groups
np.random.seed(2)
control = np.random.normal(loc=70, scale=10, size=60)
treatment = np.random.normal(loc=74, scale=12, size=55)

if SCIPY_AVAILABLE:
    tstat, pval = stats.ttest_ind(treatment, control, equal_var=False)
    print('t-stat =', round(tstat,3), 'p-value =', round(pval,4))
else:
    print('scipy not available; skipping test')

# quick effect size (Cohen's d)
pooled_sd = np.sqrt(((len(control)-1)*np.var(control, ddof=1) + (len(treatment)-1)*np.var(treatment, ddof=1)) / (len(control)+len(treatment)-2))
cohens_d = (treatment.mean() - control.mean()) / pooled_sd
print('Cohen\'s d =', round(cohens_d,3))

## 8 — Chi-square test for independence (contingency table)

Create a simple contingency table and run chi-square test.

In [ ]:
observed = np.array([[30, 10], [20, 40]])  # e.g., outcome by group
print('Observed:\n', observed)

if SCIPY_AVAILABLE:
    chi2, p, dof, expected = stats.chi2_contingency(observed)
    print('\nchi2 =', round(chi2,3))
    print('p-value =', round(p,4))
    print('dof =', dof)
    print('expected counts:\n', expected)
else:
    print('scipy not available; cannot run chi-square test')

## 9 — Simple Linear Regression (OLS)

Fit a simple linear regression (y ~ x) and interpret coefficients. We'll use numpy for OLS and also show scatter + fit line.

In [ ]:
# Simulate data with linear relationship
np.random.seed(3)
X = np.random.uniform(0, 10, size=100)
noise = np.random.normal(0, 3, size=100)
y = 2.5 * X + 5 + noise

df_lr = pd.DataFrame({'x': X, 'y': y})

# Fit using numpy (closed-form)
Xmat = np.vstack([np.ones_like(X), X]).T
beta_hat = np.linalg.inv(Xmat.T @ Xmat) @ Xmat.T @ y
intercept, slope = beta_hat[0], beta_hat[1]
print(f'intercept = {intercept:.3f}, slope = {slope:.3f}')

# Predictions & plot
y_pred = intercept + slope * X
plt.figure()
plt.scatter(X, y)
plt.plot(np.sort(X), y_pred[np.argsort(X)])
plt.xlabel('X')
plt.ylabel('y')
plt.title('Scatter with OLS fit')
plt.show()

## 10 — Exercises

1. Compute a 99% confidence interval for the sample mean of `df['score']`.
2. Perform a paired t-test: simulate before/after measurements for 30 subjects and test whether the intervention changed the mean.
3. Fit a linear regression with two predictors (add a second variable) and compare coefficients.

Try these in new cells; suggested answers are below.

## Suggested answers (brief)

1. Use `stats.t.interval(0.99, df=n-1, loc=mean, scale=se)` or z-approximation if scipy unavailable.

2. Simulate `before = np.random.normal(50, 8, 30); after = before + np.random.normal(2, 5, 30)` and run `stats.ttest_rel(after, before)`.

3. Create `X2 = X + np.random.normal(0,1,size=X.shape)` and use matrix OLS or `statsmodels` (not used here); compare coefficients and check multicollinearity.

---

### End of notebook — customize it further if you want more exercises or to include real datasets (e.g., from CSV). Good luck teaching! 🎓